# Survivorship-free + point-in-time Indian equity data — measure the bias yourself

Most Indian equity backtests are quietly inflated by two bugs that live in the **data**, not the strategy:
**survivorship bias** (delisted names dropped) and **look-ahead bias** (restated fundamentals used before they were public).

This notebook runs on the real sample CSVs and does three things on names you can verify:
1. lists the delisted names your screener silently drops,
2. reads DHFL's collapse straight from the *as-reported* fundamentals (keyed to announcement date),
3. **measures** the survivorship bias on the 99-name sample and shows it has the same sign as the full panel.

> Companion to the public headline: the same vanilla strategy run four ways on the full panel inflates by
> **+4.1 pp/yr** (survivorship + look-ahead). Here you reproduce the *mechanism* on a small, noisy slice.

In [ ]:
import glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Load the sample CSVs from wherever this runs: Kaggle mount, a local clone, or
# (on Colab) straight from the GitHub repo. pandas reads https URLs directly.
RAW = 'https://raw.githubusercontent.com/Finance-broski/pit-data-sample/main/sample_data'
def find(name):
    for pat in (f'/kaggle/input/**/{name}', f'sample_data/{name}', name):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return f'{RAW}/{name}'   # Colab / anywhere with internet

uni  = pd.read_csv(find('survivorship_universe.csv'))
fund = pd.read_csv(find('fundamentals_sample.csv'), parse_dates=['period_end', 'announce_date'])
px   = pd.read_csv(find('prices_sample.csv'), parse_dates=['date'])
print({k: v.shape for k, v in {'universe': uni, 'fundamentals': fund, 'prices': px}.items()})
print(f"{px.symbol.nunique()} priced names, {fund.symbol.nunique()} fundamentals names")

## 1. Survivorship — the names your screener silently drops

A survivorship-free universe keeps companies that delisted, merged, or went to zero, on the dates they actually traded.

In [ ]:
dead = uni[uni.status == 'inactive']
print(f"{len(dead)} delisted/suspended names out of {len(uni)} total\n")
famous = ['DHFL','JETAIRWAYS','RCOM','RELCAPITAL','IL&FSENGG','GITANJALI','RUCHISOYA','ANDHRABANK']
print(dead[dead.symbol.isin(famous)][['symbol','company_name','first_traded','last_traded']].to_string(index=False))

## 2. Point-in-time fundamentals — read DHFL's collapse as it was announced

Every figure is stamped with its **announcement date** (`announce_date`) — the day it became public, not a
restated value. `announce_lag_days` is the gap a restated screener erases (and how look-ahead bias sneaks in).

In [ ]:
dhfl = fund[fund.symbol == 'DHFL'].sort_values('period_end')
print(dhfl[['period_end','announce_date','revenue_cr','pat_cr','announce_lag_days']].to_string(index=False))
print(f"\nmedian announcement lag across the sample: {fund.announce_lag_days.median():.0f} days")
assert (fund.announce_lag_days > 0).all(), 'every figure is dated AFTER its period end - no look-ahead'
print('check passed: 0 figures dated before their period end (no look-ahead).')

## 3. Measure the survivorship bias on the sample (poke it yourself)

Same vanilla move both ways: an **equal-weight, monthly-rebalanced** basket of the priced names, 2013 on.
*Survivorship-free* holds every name on its real trading dates; *survivor-only* drops the ones now inactive —
exactly what a screener that only knows today's listed names gives you. The gap is the bias.

_This is a 99-name slice, not the universe — expect it noisier and larger than the full-panel number; that's the point of a sample._

In [ ]:
status = uni.set_index('symbol').status                          # symbol -> active/inactive
priced = px.assign(st=px.symbol.map(status)).groupby('symbol').st.first()
print('priced names by status:', priced.value_counts().to_dict())

trc = px.pivot_table(index='date', columns='symbol', values='tr_close', aggfunc='last').sort_index()
me  = trc[trc.index >= '2013-01-01'].resample('ME').last()        # month-end total-return levels
active = [s for s in me.columns if priced.get(s) == 'active']

def cagr(curve):
    curve = curve.dropna(); yrs = (curve.index[-1] - curve.index[0]).days / 365.25
    return curve.iloc[-1] ** (1/yrs) - 1
def ew_curve(cols):
    r = me[cols].pct_change(fill_method=None)                      # monthly returns; NaN where not trading
    port = r.mean(axis=1, skipna=True).fillna(0)                   # equal-weight across names live that month
    return (1 + port).cumprod()

free = ew_curve(list(me.columns))   # survivorship-free (honest)
only = ew_curve(active)             # survivor-only (naive)
g_free, g_only = cagr(free), cagr(only)
gap = (g_only - g_free) * 100
print(f"\nsurvivor-only (naive)      {g_only*100:5.1f}%/yr")
print(f"survivorship-free (honest) {g_free*100:5.1f}%/yr")
print(f"=> survivorship bias on this sample: +{gap:.1f} pp/yr")
print(f"   same SIGN as the full-panel +2.5 pp/yr (pure survivorship, 2013-2024). Small N, noisier - as expected.")

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(only.index, only.values, '--', color='#c0392b', lw=2, label=f'survivor-only (naive)  {g_only*100:.1f}%/yr')
ax.plot(free.index, free.values, '-',  color='#27ae60', lw=2.4, label=f'survivorship-free (honest)  {g_free*100:.1f}%/yr')
ax.set_yscale('log'); ax.set_ylabel('growth of Re.1 (log)')
ax.set_title(f'Survivorship bias on the 99-name sample (EW, 2013+): +{gap:.1f} pp/yr')
ax.legend(); ax.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()

## 4. The look-ahead window, quantified

Look-ahead bias is using a number before it was public. The size of the window is `announce_date - period_end`.

In [ ]:
fund['lag'] = (fund.announce_date - fund.period_end).dt.days
diff_month = (fund.announce_date.dt.to_period('M') != fund.period_end.dt.to_period('M')).mean()
print(f"median announce lag: {fund.lag.median():.0f} days")
print(f"{diff_month*100:.0f}% of filings are announced in a LATER month than the quarter they describe.")
print('Aligning fundamentals at period_end (what restated screeners effectively do) backdates')
print('those then-unknown numbers into the backtest -> look-ahead. On the full panel that one')
print('shift is worth about +4.0 pp/yr on the same value sort.')

## 5. Survivorship made visible — a delisted stock plotted to zero

A survivor-only dataset would have *removed* this line entirely, so your backtest would never have held it.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for sym, color in [('DHFL','#cc2b2b'), ('RELIANCE','#1f4e9c')]:
    s = px[px.symbol == sym].sort_values('date')
    if len(s):
        ax.plot(s.date, s.tr_close / s.tr_close.iloc[0] * 100, label=sym, color=color, lw=2)
ax.set_yscale('log'); ax.set_ylabel('Total-return index (start = 100, log)')
ax.set_title('DHFL (delisted, -> 0) vs a survivor - the line survivorship bias deletes')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Takeaway

- The universe keeps **1,209 delisted names** other sources drop -> no survivorship bias.
- Fundamentals are **announce-dated** -> no look-ahead.
- Prices are corporate-action-adjusted with a total-return series.

On this small sample the survivorship gap measures **same sign as the full panel** (full panel: +2.5 pp/yr pure
survivorship 2013-2024; +4.0 pp/yr look-ahead on a value sort; **+4.1 pp/yr combined**). Smaller N here, so noisier
and larger — a sample shows the *mechanism*, the full dataset pins the *magnitude*.

This is a **bounded free sample**. Full dataset (~3,800 names of prices, ~1,400 names of as-reported fundamentals with
full vintages, point-in-time universe membership): **https://github.com/Finance-broski/pit-data-sample** — or open an
issue to get it run on your own tickers.